# Chapter 03 제출 답안 양식. 데이터의 첫인상 읽기

> 주 제출물은 실행 완료 Notebook `chapter03/chapter03.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름:권영서
- GitHub ID:kys02101-cmd
- 작성일:2026-10-08
- 최종 제출 URL:https://github.com/kys02101-cmd/llm-data-analysis-study/blob/main/chapter03/chapter03.ipynb

## 1. 데이터 로딩과 구조 확인
### 실행/결과
- 4개 CSV 로딩 여부: 모두 TRUE
- 각 데이터 shape: 
customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)
- 주요 컬럼:

[customers]
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
['product_id', 'product_name', 'category', 'price']

[orders]
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']

- dtypes에서 주목한 컬럼:

customers.signup_date, orders.order_date: 날짜인데 str(문자열)로 읽힘
customers.age: int64. 숫자형이라 범위 확인이 필요함
products.price, order_items.unit_price: 둘 다 int64. 같은 가격 개념이 두 곳에 있음
customer_id, product_id, order_id: 모두 int64로 테이블 간 타입이 일치함
gender, city, category, payment_method, order_status: str 범주형 컬럼

### Evidence
![데이터 구조 확인](images/step01_structure.png)

### 결과 관찰
4개 CSV의 shape은 customers (150, 6), products (100, 4), orders (300, 5), order_items (764, 5)입니다. `customer_id`, `product_id`, `order_id`는 관련된 테이블에서 모두 int64로 읽혀 테이블 간 타입이 서로 일치했습니다. `signup_date`와 `order_date`는 날짜를 의미하는 컬럼인데도 str로 로딩되었습니다. `age`, `price`, `quantity`, `unit_price`는 int64로 읽혔습니다. 가격을 나타내는 컬럼이 `products.price`와 `order_items.unit_price` 두 곳에 존재한다는 점도 확인했습니다.

### 나의 해석과 판단
날짜 컬럼 두 개는 문자열 상태이므로 분석 전에 datetime으로 변환해야 하고, 변환에 실패하는 값이 없는지 먼저 확인해야 한다고 판단했습니다. `orders`와 `customers`는 `customer_id`로, `orders`와 `order_items`는 `order_id`로, `order_items`와 `products`는 `product_id`로 연결되는 구조이므로, 이 키 컬럼들의 결측, 중복, 참조 무결성을 가장 먼저 점검해야 합니다. 또한 `unit_price`와 `price`는 의미가 다를 수 있어서, 매출을 계산할 때 어느 쪽을 쓸지 정하기 전에 두 값이 일치하는지 확인해야 합니다. `age`는 숫자형이므로 0, 음수, 지나치게 큰 값처럼 비현실적인 값이 있는지 범위를 살펴봐야 합니다.

### 업무·분석적 의미
구조를 먼저 확인하지 않고 분석을 시작하면 여러 문제가 생길 수 있습니다. 문자열 상태의 날짜를 그대로 정렬하거나 집계하면 월별, 기간별 분석 결과가 틀어질 수 있습니다. 테이블 간 키 관계를 모른 채 병합하면 1:N 조인으로 행이 불어나거나 매칭되지 않는 행이 사라져서 매출과 고객 수가 왜곡될 수 있습니다. 가격 컬럼의 의미를 구분하지 않고 매출을 계산하면 실제 결제 금액과 다른 수치가 나올 수도 있습니다. 따라서 컬럼의 타입과 의미를 먼저 파악해야 이후 Chapter의 분석 결과를 신뢰할 수 있습니다.

### 한계와 추가 확인 사항
dtypes와 shape만으로는 결측치의 개수, 중복 행, 이상값 여부를 알 수 없습니다. 날짜 문자열의 형식이 일관된지, 변환할 수 없는 값이 있는지도 아직 알 수 없습니다. `order_items`의 `order_id`와 `product_id`가 각각 `orders`와 `products`에 모두 존재하는지도 확인하지 못했습니다. `order_status`, `payment_method`, `city`, `category`에 대소문자, 공백, 오타 같은 표기 불일치가 있는지도 확인이 필요합니다. 마지막으로 `unit_price`와 `price`가 실제로 같은지, `quantity`에 0이나 음수가 있는지도 추가로 확인해야 합니다.

## 2. 결측·중복·키 품질
- 주요 ID 결측: 모두 결측이 없음
- 주요 ID 중복: `order_items.order_id`는 464건이 중복, 한 주문에 여러 상품이 담기는 구조에서 나온 정상적인 값으로 해석
- 전체 행 중복: 중복 0건

![결측 중복 점검](images/step02_quality.png)

### 결과 관찰
결측 비율 요약표에서 4개 데이터셋의 모든 컬럼이 0.0%였습니다. ID 중복 점검에서는 `customers.customer_id`, `products.product_id`, `orders.order_id`의 중복이 모두 0건이었고, `order_items.order_id`만 464건으로 나타났습니다. `order_items`는 764행인데 중복 `order_id`가 464건이므로, 고유한 `order_id`는 764 − 464 = 300개입니다. 이는 `orders`의 행 수 300과 같은 숫자입니다. 전체 행 중복도 4개 데이터셋 모두 0건이었습니다.

### 나의 해석과 판단
결측 비율이 모두 0.0%이고, 각 테이블의 기본 키가 유일하며, 완전히 같은 행도 없으므로 결측 처리와 중복 제거는 우선순위가 낮다고 판단했습니다. `order_items.order_id`의 중복은 한 주문에 여러 상품이 있는 1:N 관계에서 자연스러운 값이므로 오류로 보지 않았습니다. 대신 먼저 처리할 것은 두 가지입니다. 첫째, `order_items`의 `order_id`와 `product_id`가 `orders`와 `products`에 실제로 존재하는지 확인해야 합니다. 고유 `order_id` 개수가 300개로 `orders`의 행 수와 같더라도 같은 ID 집합이라는 보장은 없기 때문입니다. 둘째, 빈 문자열, "N/A", 0처럼 결측이 아닌 값으로 채워진 경우가 없는지 범주형 빈도와 숫자 범위로 살펴봐야 합니다. 결측과 중복이 없다는 사실이 곧 데이터가 깨끗하다는 뜻은 아닙니다.

### 업무·분석적 의미
기본 키가 유일하고 중복 행이 없으면 고객 수, 상품 수, 주문 수를 행 수로 그대로 셀 수 있고, 중복 때문에 집계가 부풀려질 위험도 줄어듭니다. 반면 `order_items`는 주문 하나가 여러 행으로 나뉘어 있으므로, `orders`와 병합하거나 `order_items`의 행 수를 주문 수로 착각하면 주문 수와 매출이 부풀려질 수 있습니다. 주문 단위 분석과 상품 단위 분석을 구분해서 집계해야 합니다.

### 한계와 추가 확인 사항
결측이 `NaN`이 아닌 빈 문자열, 공백, "N/A", 0 같은 값으로 입력되어 있는지는 아직 확인하지 못했습니다. `order_items.order_item_id`의 중복 여부도 이 결과에는 포함되어 있지 않습니다. 전체 행 중복이 없다는 것은 완전히 같은 행이 없다는 뜻일 뿐, ID만 다르고 내용이 같은 사실상의 중복까지 배제하지는 못합니다. 또한 `orders.customer_id`가 `customers`에 존재하는지 같은 테이블 간 키 관계는 이 단계의 결과만으로는 알 수 없고, 섹션 4에서 따로 검증해야 합니다.

## 3. 숫자형·범주형·날짜 점검
- 숫자형 범위에서 주목한 값:`age`에 비현실적인 값 없었음. `products.price`에 0이나 음수 없었음.
- 범주형 빈도에서 주목한 값:`order_status`에서 completed 184건, cancelled 64건, refunded 52건으로 완료되지 않은 주문이 116건(38.7%)
- 날짜 변환 실패 건수: `orders.order_date`는 0건
- 날짜 범위: 2025-10-08 ~ 2026-10-07

![기본 분포와 날짜 확인](images/step03_distribution.png)

### 결과 관찰
`orders.order_date`는 변환 전 str이었고, `pd.to_datetime(errors="coerce")`로 변환한 뒤 datetime64로 바뀌었으며 변환 실패는 0건이었습니다. 가장 빠른 주문일은 2025-10-08, 가장 최근 주문일은 2026-10-07로, 약 1년 치 주문이 들어 있습니다. 범주형 컬럼의 고유값 개수는 `city` 10개, `gender` 2개, `category` 7개, `payment_method` 4개, `order_status` 3개였습니다. `city` 빈도의 합은 150으로 고객 수와 같고, `category` 빈도의 합은 100으로 상품 수와 같으며, `order_status` 빈도의 합은 300으로 주문 수와 같았습니다. 즉 이 컬럼들에는 빈 값 없이 모든 행이 분류되어 있었습니다. `order_status`의 값은 completed, cancelled, refunded 세 가지뿐이었고, 대소문자나 철자가 다른 같은 의미의 값은 보이지 않았습니다. `payment_method`는 처음 5개 행에서 card, naver_pay, bank_transfer, kakao_pay가 확인되었습니다.

### 나의 해석과 판단
날짜 변환이 모두 성공했고 주문 기간이 약 1년으로 연속적이므로, 주문 날짜는 월별·요일별 분석에 쓸 수 있다고 판단했습니다. 가장 최근 주문일이 2026-10-07이어서 미래 날짜도 없었습니다. 눈에 띄는 것은 주문의 38.7%가 completed가 아니라는 점입니다. 이것이 오류인지 업무상 가능한 값인지는 이 데이터만으로 단정할 수 없습니다. 취소와 환불이 실제로 이 정도 비율로 발생할 수도 있고, 데이터가 인위적으로 생성된 것일 수도 있기 때문입니다. 이를 구분하려면 취소·환불 주문이 `order_items`에는 어떻게 남아 있는지, 결제 수단이나 상품 카테고리에 따라 비율이 크게 다른지 더 확인해야 합니다. 또한 매출을 계산할 때 completed만 포함할지, cancelled와 refunded를 어떻게 처리할지 기준을 먼저 정해야 합니다. `city`와 `category`의 빈도 차이(예: 식품 7개)는 구성의 차이일 뿐 오류의 근거는 아니라고 보았습니다.

### 업무·분석적 의미
`order_status`를 구분하지 않고 전체 주문으로 매출이나 주문 수를 계산하면 취소·환불 건까지 포함되어 실제 매출보다 크게 계산될 수 있습니다. 날짜가 문자열인 채로 분석하면 월별 추이나 기간 비교가 틀어질 수 있으므로 datetime 변환이 먼저 필요합니다. 범주별 표본 수가 작은 값(예: 식품 7개 상품, 고양 11명)은 비율이나 평균을 비교할 때 해석이 불안정할 수 있습니다.

### 한계와 추가 확인 사항
`signup_date`의 변환 실패 여부와 범위는 아직 확인하지 못했고, 가입일이 주문일보다 늦은 고객이 있는지도 확인하지 못했습니다. `gender`와 `payment_method`의 전체 빈도는 확인하지 못했습니다. `age`, `price`, `quantity`, `unit_price`의 범위(0, 음수, 지나치게 큰 값 등)는 describe() 결과를 확인한 뒤 판단해야 합니다. 주문이 1년 치 전체에 고르게 분포하는지, 특정 달에 몰려 있는지도 아직 알 수 없습니다.

## 4. CSV 간 키 관계 검증
- 없는 `customer_id`:
- 없는 `order_id`:
- 없는 `product_id`:

![PK FK 관계 검증](images/step04_relationship.png)

### 결과 관찰

### 나의 해석과 판단
키 관계 문제가 발견되었다면 바로 삭제하면 안 되는 이유를 작성하세요.

### 업무·분석적 의미

### 한계와 추가 확인 사항

## 5. LLM 구조 설명 검증
- LLM에 제공한 Safe Context:
- LLM이 제안한 추가 점검:
- 실제 데이터에서 확인한 항목:
- 채택/수정/보류한 내용:

![LLM 구조 검토](images/step05_llm.png)

### 나의 해석과 판단
LLM 제안 중 가장 유용했던 것과 가장 조심해야 할 것을 작성하세요.

### 한계와 추가 확인 사항

## 6. Chapter 03 최종 판단
### 데이터의 첫인상 3가지
1.
2.
3.

### 다음 Chapter 전에 반드시 확인/처리해야 할 항목
1.
2.
3.

### 현재 데이터만으로 단정할 수 없는 것

## 최종 제출 체크
- [ ] Notebook을 처음부터 끝까지 실행했습니다.
- [ ] 오류 셀이 남아 있지 않습니다.
- [ ] 핵심 Evidence를 첨부했습니다.
- [ ] 관찰과 해석을 구분했습니다.
- [ ] 개인정보/Secret이 없습니다.
- [ ] `chapter03/chapter03.ipynb`가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.